> ⚠️ **Before you start:** This is a read-only course copy.
> Go to **File → Save a copy in Drive** right now, then continue working in *your* copy.
> Changes made here will not be saved.

In [ ]:
# Setup: this notebook only needs boto3, not the whole course package.
!pip install -q boto3

# Setup: AWS Credentials for Colab

## One key, stored outside the notebook

Stages 01 to 06 need nothing but a browser. Stage 07 is the first one that calls a
real model on Amazon Bedrock, so it is the first one that needs credentials.

This page is a detour, not a stage. Work through it once, and every stage from 07
onward will just run.

### The three gates

People usually assume "it didn't work" means one problem. It's almost always one of
three, and they fail at different moments:

| Gate | What it is | How it fails |
|------|-----------|--------------|
| **Credential** | A key that proves who you are | `UnrecognizedClientException`, or a silent `None` |
| **Region** | Bedrock is per-region | `NoRegionError` before anything hits the network |
| **Model access** | Each model is enabled per account, per region | `AccessDeniedException` |

The last cell in this notebook checks all three separately and tells you which one is
failing. Run it whenever something breaks later in the course.

## Part 1: Why not your AWS access keys

The obvious move is to create an IAM user, grab its access key and secret, and paste
them into the notebook. Don't. Two separate reasons, and only one of them is about
notebooks:

**A notebook is a document that gets shared.** You will send someone this link. You
will screenshot a cell. The `.ipynb` will end up in a repo. Anything typed into a cell
travels with it.

**An access key is the whole account.** It doesn't carry a scope. If it leaks, whoever
has it can do everything your IAM policy allows — and in stages 08 to 10 that policy
will include launching SageMaker training jobs. The damage is not "someone used my
model quota", it's a bill.

So we solve it twice over. **Part 2** picks a credential with a small blast radius.
**Part 5** stores it somewhere the notebook can read but the document can't carry.

## Part 2: Generate a Bedrock API key

Amazon Bedrock issues its own API keys — a single bearer token, not an access
key/secret pair. They come in two flavours:

- **Short-term** — valid up to 12 hours, inherits the permissions of whoever generated
  it. This is what AWS recommends for production.
- **Long-term** — valid until an expiry date you choose. AWS recommends these *only for
  exploration*, which is exactly what a course is.

Use a **long-term key** so you aren't regenerating it every session, and give it a short
expiry.

The thing that makes this safe enough to paste into a Google VM: **a Bedrock API key
only works on Bedrock.** It is scoped to Bedrock and Bedrock Runtime actions. It cannot
touch S3, cannot touch SageMaker, cannot read your account. Worst case, someone burns
model quota you can revoke in a click.

### Steps

1. Sign in to the [AWS console](https://console.aws.amazon.com/).
2. Go to **Amazon Bedrock** → in the left sidebar, near the bottom, **API keys**.
3. Choose the **Long-term API keys** tab → **Generate long-term API key**.
4. Set an expiry. **7 days** is plenty to get through stage 07 — pick the shortest
   window you'll actually use, you can always generate another.
5. Click generate, then **copy the key**. It's shown once. If you lose it, generate a
   new one and delete the old — there is no way to view it again.

> AWS creates an IAM user behind the scenes to carry this key, with service-specific
> credentials limited to Bedrock. You don't need to manage that user; deleting the key
> is enough.

## Part 3: Pick a region

Bedrock is regional. The models available, and whether you've enabled them, are both
per-region. Pick one and use it for the rest of the course.

`us-east-1` is the safe default — it gets models first and has the widest selection.

Write down whichever you pick. You need it in Part 5, and a mismatch between your region
and your model ID is the single most common reason the verify cell fails.

## Part 4: Enable model access

A fresh AWS account cannot call any Bedrock model until you ask for it. This catches
almost everyone, because the key works fine and the failure looks like a permissions bug.

1. In the Bedrock console, confirm you're in the region you chose in Part 3
   (top-right region selector).
2. Left sidebar → **Model access**.
3. **Modify model access** → tick the Anthropic Claude models → **Next** → **Submit**.

Access for Claude models is usually granted immediately. The **Model access** page shows
each model as *Access granted* when it's ready.

Do this in the same region as Part 3. Access granted in `us-east-1` does nothing for
`eu-west-1`.

## Part 5: Store the key in Colab Secrets

Colab has a built-in secret store. It lives in your Google account, **not** in the
notebook file — so the key survives sharing the notebook, and doesn't travel with it.

1. In the Colab sidebar, click the **🔑 key icon** (*Secrets*).
2. **+ Add new secret**. Name it exactly `AWS_BEARER_TOKEN_BEDROCK`, paste the key from
   Part 2 as the value.
3. Add a second secret named `AWS_DEFAULT_REGION`, value = the region from Part 3
   (e.g. `us-east-1`).
4. **Toggle "Notebook access" on for both.** This is per notebook and it is off by
   default. If you skip it, the next cell fails with `NotebookAccessError` even though
   the secrets exist.

Set them once and every notebook in the course can use them — you just flip the
Notebook access toggle for each new one.

### Loading them

The names matter. Those two are exactly what boto3 looks for in the environment, so once
they're loaded, no code anywhere else in the course has to know about credentials.

In [ ]:
import os

def load_aws_secrets():
    """Copy Colab Secrets into environment variables.

    boto3 reads AWS_BEARER_TOKEN_BEDROCK and AWS_DEFAULT_REGION from the
    environment on its own, so after this runs every later cell just works —
    no client is ever handed a credential explicitly.
    """
    try:
        from google.colab import userdata
    except ImportError:
        print("Not running in Colab — using whatever credentials this machine already has.")
        return

    for name in ("AWS_BEARER_TOKEN_BEDROCK", "AWS_DEFAULT_REGION"):
        try:
            os.environ[name] = userdata.get(name)
            print(f"loaded  {name}")
        except Exception as e:
            # SecretNotFoundError  -> the secret does not exist (check the spelling)
            # NotebookAccessError  -> it exists, but this notebook is not allowed to read it
            print(f"MISSING {name}  ({type(e).__name__})")


load_aws_secrets()

## Part 6: Verify

This cell checks the three gates in order and stops at the first one that fails, so the
message you get points at one problem instead of a stack trace.

Run it now. Run it again any time a later stage stops working.

In [ ]:
import os

import boto3
import botocore.exceptions

# Bedrock reaches Claude through a cross-region inference profile, and the prefix
# has to match your region: us-east-1 -> "us.", eu-west-1 -> "eu.", and so on.
BASE_MODEL = "anthropic.claude-sonnet-4-6"

# What each Bedrock error actually means, in the order you are likely to hit them.
REMEDIES = {
    "UnrecognizedClientException": "The key is not valid. Regenerate it (Part 2) and update the secret.",
    "InvalidSignatureException": "The key is malformed — likely a partial copy/paste. Re-copy it.",
    "AccessDeniedException": "The key is valid but the model is not enabled in this region. See Part 4.",
    "ValidationException": "This model ID is not available in this region. Check the region prefix.",
    "ResourceNotFoundException": "No such model in this region. Check Part 3 and Part 4 agree.",
    "ThrottlingException": "Rate limited by Bedrock. Wait a moment and run this again.",
}


def verify():
    # Gate 1 — is there a credential at all?
    if not os.environ.get("AWS_BEARER_TOKEN_BEDROCK"):
        print("FAIL  no API key in the environment.")
        print("      Run the cell in Part 5. If it printed MISSING, the secret is not")
        print("      set or Notebook access is off for it.")
        return False
    print("ok    API key is set")

    # Gate 2 — Bedrock is regional, and boto3 fails before the network without one.
    region = os.environ.get("AWS_DEFAULT_REGION")
    if not region:
        print("FAIL  no region set. Add an AWS_DEFAULT_REGION secret (Part 3).")
        return False
    model_id = f"{region.split('-')[0]}.{BASE_MODEL}"
    print(f"ok    region is {region}, so the model ID is {model_id}")

    # Gate 3 — the only real proof is a call that comes back.
    try:
        client = boto3.client("bedrock-runtime", region_name=region)
        response = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": "Reply with the single word: ready"}]}],
        )
    except botocore.exceptions.ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"FAIL  Bedrock rejected the call: {code}")
        print(f"      {REMEDIES.get(code, e.response['Error'].get('Message', ''))}")
        return False
    except botocore.exceptions.BotoCoreError as e:
        print(f"FAIL  could not reach Bedrock: {type(e).__name__}: {e}")
        return False

    reply = response["output"]["message"]["content"][0]["text"].strip()
    usage = response["usage"]
    print(f"ok    Claude replied: {reply!r}")
    print(f"      ({usage['inputTokens']} in / {usage['outputTokens']} out — a fraction of a cent)")
    print()
    print("All three gates open. Stage 07 will run.")
    return True


verify()

## Part 7: Housekeeping

**Cost.** Everything in stage 07 costs a fraction of a cent. The verify call above is a
handful of tokens. This is not a course where you need to watch a bill — but set a
[budget alert](https://console.aws.amazon.com/billing/home#/budgets) anyway, because
that's the habit, not because this will trip it.

**Rotation.** The key expires on the date you set in Part 2. When it does, generate a new
one and update the Colab secret — nothing else changes.

**Revoking.** Bedrock console → **API keys** → select the key → delete. It stops working
immediately. Do this the moment you suspect it's out of your hands, then generate a
replacement; there is no partial disable.

**If you accidentally paste it into a cell.** Delete the cell, then *still revoke the
key*. Colab keeps outputs and checkpoints, and if you shared the notebook at any point
the value is already gone. Revoking takes ten seconds; deciding whether the leak
mattered takes longer than that and you'll get it wrong.

## What about stages 08 to 10?

A Bedrock API key is deliberately narrow — Bedrock and nothing else. That's what makes
it safe here, and it's also why it won't carry you through the rest of the course. S3,
SageMaker, and MLflow all need credentials with real account access.

That changes the calculation. A credential that can start SageMaker training jobs is not
something to keep in a browser VM you don't control, and the honest answer is that the
"run it all in Colab" model runs out at stage 07. Stages 08 to 10 move to your own AWS
account with session credentials, and those stages will cover that setup when they land.

For now: you have what stage 07 needs.

---

**Next:** [Stage 07 — LLM Integration](https://colab.research.google.com/github/marceloacosta/churn-prediction-pipeline/blob/main/modules/07-llm-integration/07-llm-integration.ipynb)

*Reference: [Bedrock API keys](https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys.html), [using an API key](https://docs.aws.amazon.com/bedrock/latest/userguide/api-keys-use.html)*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*